## RAF-DB analysis and setup for tests:

Mapping the number to the emotion based on the paper:

In [3]:
from pathlib import Path

In [6]:
root = Path(r"DATASET")  

label_map = {
    "1": "surprise",
    "2": "fear",
    "3": "disgust",
    "4": "happiness",
    "5": "sadness",
    "6": "anger",
    "7": "neutral",
}

for split in ["train", "test"]:
    split_dir = root / split
    for old_label, new_label in label_map.items():
        old_path = split_dir / old_label
        new_path = split_dir / new_label

        # só renomeia se a pasta antiga existir
        if old_path.exists() and old_path.is_dir():
            # se já existir a nova pasta, evita sobrescrever
            if new_path.exists():
                print(f"[WARNING] {new_path} already exists, not overwriting.")
            else:
                old_path.rename(new_path)
                print(f"Renamed: {old_path} -> {new_path}")

print("\nRenaming completed.\n")


Renaming completed.



Getting the amount of data for each class:

In [7]:
for split in ["train", "test"]:
    split_dir = root / split
    print(f"=== {split} ===")
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            n_files = sum(1 for f in class_dir.iterdir() if f.is_file())
            print(f"{class_dir.name}: {n_files} files")
    print()

=== train ===
anger: 705 files
disgust: 717 files
fear: 281 files
happiness: 4772 files
neutral: 2524 files
sadness: 1982 files
surprise: 1290 files

=== test ===
anger: 162 files
disgust: 160 files
fear: 74 files
happiness: 1185 files
neutral: 680 files
sadness: 478 files
surprise: 329 files



As we can see, in the test subset we have very few examples of each class (such as fear and anger). As we are not going to perform any training, I opted to merge this 2 subsets, so we have a bigger test set.

In [8]:
from pathlib import Path
import shutil

train_dir = root / "train"
test_dir = root / "test"
out_dir = root / "all_test"

out_dir.mkdir(exist_ok=True)

for class_dir in train_dir.iterdir():
    if not class_dir.is_dir():
        continue

    class_name = class_dir.name
    print(f"Merging class: {class_name}")

    out_class_dir = out_dir / class_name
    out_class_dir.mkdir(parents=True, exist_ok=True)

    for f in class_dir.iterdir():
        if f.is_file():
            dest = out_class_dir / f"train_{f.name}"
            shutil.copy2(f, dest)

    test_class_dir = test_dir / class_name
    if test_class_dir.exists() and test_class_dir.is_dir():
        for f in test_class_dir.iterdir():
            if f.is_file():
                dest = out_class_dir / f"test_{f.name}"
                shutil.copy2(f, dest)
    else:
        print(f"Warning: test folder not found for class {class_name}")

print("Completed! Combined data in all_test.")

Merging class: anger
Merging class: disgust
Merging class: fear
Merging class: happiness
Merging class: neutral
Merging class: sadness
Merging class: surprise
Completed! Combined data in all_test.


In [9]:
for split in ["all_test"]:
    split_dir = root / split
    print(f"=== {split} ===")
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            n_files = sum(1 for f in class_dir.iterdir() if f.is_file())
            print(f"{class_dir.name}: {n_files} files")
    print()

=== all_test ===
anger: 867 files
disgust: 877 files
fear: 355 files
happiness: 5957 files
neutral: 3204 files
sadness: 2460 files
surprise: 1619 files



In [10]:
shutil.rmtree(root / "train"); shutil.rmtree(root / "test")

In [11]:
all_test_dir = root / "all_test"
new_all_test_dir = root.parent / "all_test"

all_test_dir.rename(new_all_test_dir)
shutil.rmtree(root)

In [10]:
import pandas as pd
from pathlib import Path

original_path = Path("all_test")

rows = []

for emotion_dir in original_path.iterdir():
    print(emotion_dir)
    if emotion_dir.is_dir():  
        img_paths = list(emotion_dir.glob("*.jpg"))
        for img_path in img_paths:
            emotion = emotion_dir.name
            rows.append({
                "emotion": emotion,
                "image_path": str(img_path)
            })

df = pd.DataFrame(rows, columns=["emotion", "image_path"])
print(df.head())
print(len(df), "linhas criadas")


all_test\anger
all_test\disgust
all_test\fear
all_test\happiness
all_test\neutral
all_test\sadness
all_test\surprise
  emotion                                 image_path
0   anger  all_test\anger\test_test_0017_aligned.jpg
1   anger  all_test\anger\test_test_0027_aligned.jpg
2   anger  all_test\anger\test_test_0037_aligned.jpg
3   anger  all_test\anger\test_test_0042_aligned.jpg
4   anger  all_test\anger\test_test_0057_aligned.jpg
15339 linhas criadas


In [11]:
df

,emotion,image_path
0,anger,all_test\anger\test_test_0017_aligned.jpg
1,anger,all_test\anger\test_test_0027_aligned.jpg
2,anger,all_test\anger\test_test_0037_aligned.jpg
3,anger,all_test\anger\test_test_0042_aligned.jpg
4,anger,all_test\anger\test_test_0057_aligned.jpg
...,...,...
15334,surprise,all_test\surprise\train_train_09135_aligned.jpg
15335,surprise,all_test\surprise\train_train_09145_aligned.jpg
15336,surprise,all_test\surprise\train_train_09147_aligned.jpg
15337,surprise,all_test\surprise\train_train_09151_aligned.jpg


In [12]:
df.to_csv("dataset_path_labels.csv", index=False)